In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
import random
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, BatchNormalization, ReLU, Concatenate, GRU, Dense, Add, Conv1D, GlobalAveragePooling1D
from tensorflow.keras.optimizers import Adam
from imblearn.over_sampling import SMOTE
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Load dataset
df = pd.read_csv('./dataset/UCI_Credit_Card.csv')

In [3]:
df.shape

(30000, 25)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ID                          30000 non-null  int64  
 1   LIMIT_BAL                   30000 non-null  float64
 2   SEX                         30000 non-null  int64  
 3   EDUCATION                   30000 non-null  int64  
 4   MARRIAGE                    30000 non-null  int64  
 5   AGE                         30000 non-null  int64  
 6   PAY_0                       30000 non-null  int64  
 7   PAY_2                       30000 non-null  int64  
 8   PAY_3                       30000 non-null  int64  
 9   PAY_4                       30000 non-null  int64  
 10  PAY_5                       30000 non-null  int64  
 11  PAY_6                       30000 non-null  int64  
 12  BILL_AMT1                   30000 non-null  float64
 13  BILL_AMT2                   300

In [5]:
df['default.payment.next.month'].value_counts()

default.payment.next.month
0    23364
1     6636
Name: count, dtype: int64

In [6]:
# Feature & Target
X = df.drop(columns=['ID', 'default.payment.next.month'])
y = df['default.payment.next.month']

In [7]:
# === Step 3: Apply SMOTE BEFORE splitting ===
sm = SMOTE(random_state=42)
X_balanced, y_balanced = sm.fit_resample(X, y)

# === Step 4: Verify new class distribution ===
print("After SMOTE:", Counter(y_balanced))  # Expect equal class distribution

After SMOTE: Counter({1: 23364, 0: 23364})


In [8]:
# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_balanced)

In [9]:
# === Jaya Algorithm ===
def jaya_hyperparams(population_size=10, iterations=5):
    param_bounds = {
        'learning_rate': (0.0001, 0.01),
        'gru_units': (32, 128),
        'batch_size': (16, 64),
        'epochs': (10, 50)
    }

    def random_solution():
        return {
            'learning_rate': np.random.uniform(*param_bounds['learning_rate']),
            'gru_units': int(np.random.uniform(*param_bounds['gru_units'])),
            'batch_size': int(np.random.uniform(*param_bounds['batch_size'])),
            'epochs': int(np.random.uniform(*param_bounds['epochs']))
        }

    def fitness(solution, X, y):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
        X_train_rnn = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
        X_test_rnn = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

        model = build_rxt_gru((X_train.shape[1], 1), solution['gru_units'])
        model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=solution['learning_rate']), metrics=['accuracy'])
        model.fit(X_train_rnn, y_train, epochs=solution['epochs'], batch_size=solution['batch_size'], verbose=0)
        _, acc = model.evaluate(X_test_rnn, y_test, verbose=0)
        return acc

    population = [random_solution() for _ in range(population_size)]
    best = max(population, key=lambda sol: fitness(sol, jaya_X, jaya_y))
    worst = min(population, key=lambda sol: fitness(sol, jaya_X, jaya_y))

    for _ in range(iterations):
        new_population = []
        for sol in population:
            new_sol = {}
            for key in sol:
                r1, r2 = random.random(), random.random()
                new_val = sol[key] + r1 * (best[key] - abs(sol[key])) - r2 * (worst[key] - abs(sol[key]))
                low, high = param_bounds[key]
                new_val = max(low, min(high, new_val))
                new_sol[key] = int(new_val) if 'int' in str(type(param_bounds[key][0])) else new_val
            new_population.append(new_sol)
        population = new_population
        best = max(population, key=lambda sol: fitness(sol, jaya_X, jaya_y))
        worst = min(population, key=lambda sol: fitness(sol, jaya_X, jaya_y))

    return best

In [10]:
# === RXT-GRU Model Builder ===
def build_rxt_gru(input_shape, gru_units):
    inputs = Input(shape=input_shape)
    convs = [Conv1D(32, kernel_size=3, padding='same', activation='relu')(inputs) for _ in range(4)]
    x = Add()(convs)
    x = GRU(gru_units, return_sequences=True)(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(1, activation='sigmoid')(x)
    return Model(inputs, x)

# These global placeholders are required for Jaya's fitness function.
jaya_X, jaya_y = None, None

In [11]:
def build_resnet_model(input_shape):
    inp = Input(shape=input_shape)
    x = Dense(64, activation='relu')(inp)
    x = BatchNormalization()(x)
    x = Dense(64, activation='relu')(x)
    x = BatchNormalization()(x)
    shortcut = Dense(64)(inp)
    x = Add()([x, shortcut])
    x = Dense(32, activation='relu')(x)
    out = Dense(1, activation='sigmoid')(x)
    return Model(inputs=inp, outputs=out)

In [12]:
def evaluate_with_rxt_gru(X_data, y_data, with_smote=True):
    global jaya_X, jaya_y

    max_components = min(X_data.shape[0], X_data.shape[1])
    results = {}

    if with_smote:
        sm = SMOTE(random_state=42)
        X_data, y_data = sm.fit_resample(X_data, y_data)

    possible_components = [19, 21, 35, 38, 41]
    components = [n for n in possible_components if n <= max_components]

    if not components:
        raise ValueError(f"No valid components. Max available is {max_components}")

    print(f"Evaluating with components: {components}")

    for n_components in components:
        pca = PCA(n_components=n_components)
        X_pca = pca.fit_transform(X_data)
        X_train, X_test, y_train, y_test = train_test_split(X_pca, y_data, test_size=0.3, random_state=42)

        # Save for Jaya
        jaya_X, jaya_y = X_train, y_train

        # === AdaBoost ===
        ada = AdaBoostClassifier(n_estimators=50)
        ada.fit(X_train, y_train)
        ada_preds = ada.predict(X_test)
        ada_acc = accuracy_score(y_test, ada_preds) * 100

        # === SVM ===
        svm = SVC(kernel='rbf', gamma='scale')
        svm.fit(X_train, y_train)
        svm_preds = svm.predict(X_test)
        svm_acc = accuracy_score(y_test, svm_preds) * 100

        # === RXT-GRU with Jaya ===
        params = jaya_hyperparams()
        X_train_rnn = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
        X_test_rnn = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

        rxt_gru = build_rxt_gru((X_train.shape[1], 1), params['gru_units'])
        rxt_gru.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=params['learning_rate']), metrics=['accuracy'])
        rxt_gru.fit(X_train_rnn, y_train, epochs=params['epochs'], batch_size=params['batch_size'], verbose=0)
        _, rxt_acc = rxt_gru.evaluate(X_test_rnn, y_test, verbose=0)

        # === ResNet ===
        X_train_bench, X_test_bench, y_train_bench, y_test_bench = train_test_split(X_pca, y_data, test_size=0.3, random_state=42)
        resnet_model = build_resnet_model((X_pca.shape[1],))
        resnet_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        resnet_model.fit(X_train_bench, y_train_bench, epochs=10, batch_size=32, verbose=0)
        preds_resnet = resnet_model.predict(X_test_bench)
        preds_resnet_label = (preds_resnet > 0.5).astype(int)
        resnet_acc = accuracy_score(y_test_bench, preds_resnet_label) * 100

        # Store results and models
        results[n_components] = {
            'accuracy': {
                'ADA': ada_acc,
                'SVM': svm_acc,
                'Proposed RXT-J': rxt_acc * 100,
                'ResNet': resnet_acc
            },
            'models': {
                'ADA': ada,
                'SVM': svm,
                'RXT-GRU': rxt_gru,
                'ResNet': resnet_model
            },
            'PCA': pca
        }

    return results

In [13]:
# Run evaluations
print("Evaluating with SMOTE...")
with_smote_results = evaluate_with_rxt_gru(X_scaled, y_balanced, with_smote=True)

Evaluating with SMOTE...
Evaluating with components: [19, 21]
439/439 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
439/439 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step   


In [14]:
print(with_smote_results)

{19: {'accuracy': {'ADA': 71.93808402881803, 'SVM': 75.36914187887866, 'Proposed RXT-J': 75.34061074256897, 'ResNet': 76.3535202225551}, 'models': {'ADA': AdaBoostClassifier(), 'SVM': SVC(), 'RXT-GRU': <Functional name=functional_120, built=True>, 'ResNet': <Functional name=functional_121, built=True>}, 'PCA': PCA(n_components=19)}, 21: {'accuracy': {'ADA': 72.36607461302518, 'SVM': 75.46187317212355, 'Proposed RXT-J': 75.56173801422119, 'ResNet': 76.73157857193809}, 'models': {'ADA': AdaBoostClassifier(), 'SVM': SVC(), 'RXT-GRU': <Functional name=functional_242, built=True>, 'ResNet': <Functional name=functional_243, built=True>}, 'PCA': PCA(n_components=21)}}


In [ ]:
print("\nEvaluating without SMOTE...")
without_smote_results = evaluate_with_rxt_gru(X_scaled, y_balanced, with_smote=False)


Evaluating without SMOTE...
Evaluating with components: [19, 21]


In [ ]:
print(without_smote_results)